# 4. Scan operators

A **scan** computes the levels of a field one after the other along an axis, e.g. the vertical: each level depends on the previous one, through a *carry*. Typical uses: vertical integrals, tridiagonal solvers, accumulations.

Contents: scan definitions and operators · scans in field operators · requested domains and mesh columns · scans on the backends.

## Scan definitions and operators

| Concept | Kind | Meaning |
|---|---|---|
| `ScanOperatorDefinition[**P, R]` | protocol | A definition following `SCAN_OPERATOR_RULES`: it takes the carry and the values of one level, scalars, and returns the new carry, which is also the value of the result on that level |
| `ScanConfig` | dataclass | How to scan: along `axis`, `forward` or backward, starting from the carry `init` |
| `ScanOperator[**P, R]` | protocol | A `DSLOperator` of kind `SCAN_OPERATOR`, with its `__gt_scan_config__`. It takes and returns *fields*, so its signature differs from its definition's |
| `CompiledScanOperator` | protocol | A `ScanOperator` that can also be compiled ahead of time (see "6. Compilation") |

`@scan_operator(axis=K, forward=True, init=0.0)` checks the definition and wraps it into a `ScanOperatorWrapper`, which runs the definition level by level, for all the columns at once (embedded execution). The domain of the result is computed forward: the intersection of the domains of the arguments. The extents of a scan operator read each argument along the scan axis, `(ScanAlong(axis),)`: backward, a level needs all the previous ones, so the whole axis is needed.

In [1]:
import numpy as np
from concepts import Axis, DSLOperatorKind, Location, ScanConfig, is_scan_operator_definition
from implementation import NumPyArrayField, scan_operator


class Cell(Location): ...


class K(Axis): ...  # vertical levels, from the top (K(0)) down


@scan_operator(axis=K, forward=True, init=0.0)
def pressure_below(above: float, weight: float) -> float:
    """The pressure at the bottom of a layer: the pressure above plus its weight."""
    return above + weight


@scan_operator(axis=K, forward=False, init=0.0)
def height_above_ground(below: float, thickness: float) -> float:
    return below + thickness


@scan_operator(axis=K, init=1.0)
def decay(previous: float, rate: float) -> float:
    return previous * (1.0 - rate)


config = pressure_below.__gt_scan_config__
print(
    "kind:",
    pressure_below.__gt_operator_kind__ is DSLOperatorKind.SCAN_OPERATOR,
    "| config:",
    config == ScanConfig(axis=K),
    config.axis.__name__,
)
print(
    "definition:",
    is_scan_operator_definition(pressure_below.__gt_definition__),
    "| extents:",
    pressure_below.__gt_extents__,
)

weight = NumPyArrayField(
    {Cell: range(0, 3), K: range(0, 4)},
    [[1.0, 2.0, 3.0, 4.0], [2.0, 2.0, 2.0, 2.0], [0.0, 1.0, 0.0, 1.0]],
)
print("pressure below      =", pressure_below(weight))
print("height above ground =", height_above_ground(weight))  # backward: from the bottom up
print(
    "decay               =",
    decay(NumPyArrayField({Cell: range(0, 1), K: range(0, 4)}, [[0.5, 0.5, 0.5, 0.5]])),
)

kind: True | config: True K
definition: True | extents: {'weight': [(ScanAlong(axis=<class '__main__.K'>),)]}
pressure below      = Field(Cell[0:3], K[0:4]: [[1.0, 3.0, 6.0, 10.0], [2.0, 4.0, 6.0, 8.0], [0.0, 1.0, 1.0, 2.0]])
height above ground = Field(Cell[0:3], K[0:4]: [[10.0, 9.0, 7.0, 4.0], [8.0, 6.0, 4.0, 2.0], [2.0, 2.0, 1.0, 1.0]])
decay               = Field(Cell[0:1], K[0:4]: [[0.5, 0.25, 0.125, 0.0625]])


## Scans in field operators

A scan operator can be called from a field operator, like any other DSL operator; the field operator then combines the scanned fields point by point. The arguments of a scan operator can be fields with different domains: the scan runs on their intersection.

In [2]:
from concepts import Dims, Field
from implementation import field_operator


@field_operator()
def mid_level_pressure(
    weight: Field[Dims[Cell, K], float], surface: float
) -> Field[Dims[Cell, K], float]:
    return surface + pressure_below(weight) - 0.5 * weight  # at the middle of each layer


@scan_operator(axis=K)
def pressure_of_layers(above: float, density: float, thickness: float) -> float:
    return above + density * thickness


print("mid-level pressure  =", mid_level_pressure(weight, 1000.0))
density = NumPyArrayField({Cell: range(0, 3), K: range(0, 2)}, np.ones((3, 2)))  # 2 levels
print("on the intersection =", pressure_of_layers(density, weight))  # only K[0:2]

mid-level pressure  = Field(Cell[0:3], K[0:4]: [[1000.5, 1002.0, 1004.5, 1008.0], [1001.0, 1003.0, 1005.0, 1007.0], [1000.0, 1000.5, 1001.0, 1001.5]])
on the intersection = Field(Cell[0:3], K[0:2]: [[1.0, 3.0], [2.0, 4.0], [0.0, 1.0]])


## Requested domains and mesh columns

As for field operators, `op[domain](...)` restricts the result to a requested domain. On a mesh, the horizontal dimension is unstructured: a scan can run on the columns of an index set of cells, e.g. those of a region, and its result keeps that index set.

In [3]:
print("bottom two levels =", pressure_below[{Cell: range(0, 3), K: range(2, 4)}](weight))

region = weight[{Cell: (0, 2)}]  # the columns of cells 0 and 2 only
print("region columns    =", pressure_below(region))
print("one level of them =", pressure_below[{Cell: (2,), K: range(3, 4)}](region))

bottom two levels = Field(Cell[0:3], K[2:4]: [[6.0, 10.0], [6.0, 8.0], [1.0, 2.0]])
region columns    = Field(Cell{0, 2}, K[0:4]: [[1.0, 3.0, 6.0, 10.0], [0.0, 1.0, 1.0, 2.0]])
one level of them = Field(Cell{2}, K[3:4]: [[2.0]])


## Scans on the backends

Scan operators are re-targeted like field operators: `backend(scan)` dispatches to `backend.scan_operator(scan)`. All the backends of `implementation.py` compute scans eagerly, level by level: `eager`, `lazy` and `lazy_backward` with an `EagerScanOperator`, `column_major` storing the result in its layout, and `compiled` running generated code for each level (see "5. Processors and backends" and "6. Compilation"). A field operator calling a scan runs on a backend too.

In [4]:
from implementation import compiled, eager, lazy, lazy_backward

for backend in (eager, lazy, compiled):
    print(f"--- {backend}: {type(backend(pressure_below)).__name__}")
    column = backend(pressure_below)[{Cell: (1,), K: range(0, 4)}](weight)
    print("cell 1 =", column)
print("--- a field operator calling the scan, on eager")
mid = eager(mid_level_pressure)(weight, 1000.0)
print("--- on lazy_backward, for cell 1, levels 2-3")
# the scan needs all the levels above, from K(0): only the cell is restricted backward
print(lazy_backward(mid_level_pressure)[{Cell: (1,), K: range(2, 4)}](weight, 1000.0))

--- eager: EagerScanOperator
[scan] computed Field(Cell[0:3], K[0:4]: [[1.0, 3.0, 6.0, 10.0], [2.0, 4.0, 6.0, 8.0], [0.0, 1.0, 1.0, 2.0]])
cell 1 = Field(Cell{1}, K[0:4]: [[2.0, 4.0, 6.0, 8.0]])
--- lazy: EagerScanOperator
[scan] computed Field(Cell[0:3], K[0:4]: [[1.0, 3.0, 6.0, 10.0], [2.0, 4.0, 6.0, 8.0], [0.0, 1.0, 1.0, 2.0]])
cell 1 = Field(Cell{1}, K[0:4]: [[2.0, 4.0, 6.0, 8.0]])
--- compiled: GeneratedScanOperator
[compiled] compiling pressure_below
[compiled] computed Field(Cell[0:3], K[0:4]: [[1.0, 3.0, 6.0, 10.0], [2.0, 4.0, 6.0, 8.0], [0.0, 1.0, 1.0, 2.0]])
cell 1 = Field(Cell{1}, K[0:4]: [[2.0, 4.0, 6.0, 8.0]])
--- a field operator calling the scan, on eager
[eager] computed Field(Cell[0:3], K[0:4]: [[1000.5, 1002.0, 1004.5, 1008.0], [1001.0, 1003.0, 1005.0, 1007.0], [1000.0, 1000.5, 1001.0, 1001.5]])
--- on lazy_backward, for cell 1, levels 2-3
[lazy_backward] computed Cell{1}, K[2:4]
Field(Cell{1}, K[2:4]: [[1005.0, 1007.0]])
